In [4]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf, bds
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan, breaks_cusumolsresid
from statsmodels.tsa.seasonal import STL
from statsmodels.formula.api import ols
from statsmodels.tsa.stattools import acf
from scipy.stats import jarque_bera
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import mutual_info_regression
from arch.unitroot import VarianceRatio
from arch import arch_model
import antropy as ant
import hurst
from nolds import lyap_r



In [5]:
df = pd.read_csv("crude_oil_2007_to_present.csv")
# Remove 2nd and 3rd rows (Python uses zero-based indexing)
df = df.drop([1, 2]).reset_index(drop=True)

# Optional: verify
print(df.head(5))

        Price               Close                High                 Low  \
0      Ticker                CL=F                CL=F                CL=F   
1  2007-01-03   58.31999969482422  60.970001220703125  57.720001220703125   
2  2007-01-04   55.59000015258789   58.58000183105469   55.38999938964844   
3  2007-01-05  56.310001373291016  56.400001525878906  54.900001525878906   
4  2007-01-08   56.09000015258789  57.720001220703125  55.099998474121094   

                 Open  Volume  
0                CL=F    CL=F  
1   60.95000076293945  248547  
2  58.349998474121094  324010  
3   55.40999984741211  250130  
4    56.2400016784668  298727  


In [10]:
# Research review: The canonical CSV order is Date, Close, High, Low, Open, Volume. This manual relabelling can swap the meanings of Open and Close. Backfilling also uses later observations. Retained for review, not silently corrected.
# Assuming your DataFrame is named df
df = df.drop(index=0).reset_index(drop=True)

# Optional: confirm it worked
print(df.head())
df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']  # rename manually
for col in df.columns:
    if col != 'Date':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Optional: fill missing values created by coercion
df = df.fillna(method='ffill').fillna(method='bfill')

# Confirm that everything is now numeric
print(df.dtypes)

         Date                Open                High                 Low  \
0  2007-01-09   55.63999938964844   56.20000076293945  53.880001068115234   
1  2007-01-10   54.02000045776367  55.810001373291016  53.439998626708984   
2  2007-01-11  51.880001068115234   54.72999954223633   51.79999923706055   
3  2007-01-12    52.9900016784668   53.11000061035156  51.560001373291016   
4  2007-01-16  51.209999084472656   53.54999923706055  50.529998779296875   

                Close  Volume  
0  56.119998931884766  333330  
1               55.75  282508  
2   53.59000015258789  341657  
3   52.06999969482422  317900  
4  52.939998626708984  276095  
Date       object
Open      float64
High      float64
Low       float64
Close     float64
Volume      int64
dtype: object


C:\Users\ASUS\AppData\Local\Temp\ipykernel_73000\3591809582.py:12: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill').fillna(method='bfill')


In [2]:
# Research review: This diagnostic uses a different DF1-DF4 boundary scheme from the tuned notebooks; report exact years with every metric.
"""
Dataset diagnostic script — run this on each of your four CSVs before trusting
any claim about "Gold is smooth", "Crude Oil is shock-driven", or the DF1-DF4
regime boundaries. It replaces assumption with actual measurement:

  1. Real date range and row count (does the file really go "to present"?).
  2. Gaps in the date index (missing trading days / data holes).
  3. How many observations actually fall in each of your DF1-DF4 regimes.
  4. Objective volatility/character metrics per regime, so "smooth trend" vs.
     "shock-driven" is backed by numbers instead of visual impression:
       - daily return mean, std, skewness, kurtosis (fat tails = shocks)
       - lag-1 autocorrelation of returns (higher = more persistent/trendy,
         lower/negative = more mean-reverting/jumpy)
       - max single-day move (the biggest "shock" actually present)
  5. A quick check for the specific concern raised earlier: does the price
     level in the data match what the write-up assumed?

Usage: set CSV_PATHS below to your four files, then run.
"""

import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. POINT THIS AT YOUR ACTUAL FILES
# ------------------------------------------------------------------
CSV_PATHS = {
    "Gold":      "gold_2007_to_present.csv",
    "Forex":     "forex_usd_inr_2007_to_present.csv",     # adjust filename
    "NIFTY 50":  "nifty50_2007_to_now.csv",
    "Crude Oil": "crude_oil_2007_to_present.csv",      # adjust filename
}

# The regime boundaries as currently used in the notebook (cells 12/17/20/21/22)
REGIMES = {
    "DF1": (2007, 2011),
    "DF2": (2012, 2019),
    "DF3": (2020, 2022),
    "DF4": (2023, None),   # None = open-ended, up to max year in data
}


def load_and_clean(path):
    df = pd.read_csv(path)
    df = df.rename(columns={df.columns[0]: "Date"})
    date_col = None
    for c in ["Date", "date", "DATE", "Timestamp", "timestamp"]:
        if c in df.columns:
            date_col = c
            break
    if date_col is None:
        raise ValueError(f"No date column found in {path}. Columns: {df.columns.tolist()}")

    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).sort_values(date_col)
    df = df.drop_duplicates(subset=[date_col], keep="first").reset_index(drop=True)

    for col in ["Open", "High", "Low", "Close"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.rename(columns={date_col: "Date"})
    return df.dropna(subset=["High", "Low", "Close"]).reset_index(drop=True)


def regime_slice(df, start_year, end_year):
    mask = df["Date"].dt.year >= start_year
    if end_year is not None:
        mask &= df["Date"].dt.year <= end_year
    return df[mask]


def character_stats(df):
    """Objective measures of 'smooth trend' vs 'shock-driven', in place of eyeballing."""
    if len(df) < 10:
        return None
    returns = df["Close"].pct_change().dropna()
    if len(returns) < 5:
        return None
    return {
        "n_obs": len(df),
        "mean_daily_return_%": returns.mean() * 100,
        "std_daily_return_%": returns.std() * 100,
        "skewness": returns.skew(),
        "kurtosis": returns.kurt(),          # >0 = fatter tails than normal = more shock-prone
        "lag1_autocorr": returns.autocorr(lag=1),  # higher = more trend-persistent
        "max_up_move_%": returns.max() * 100,
        "max_down_move_%": returns.min() * 100,
        "price_min": df["Close"].min(),
        "price_max": df["Close"].max(),
    }


def check_date_gaps(df, expected_freq_days=1):
    """Flags unusually large gaps between consecutive observations (data holes)."""
    diffs = df["Date"].diff().dt.days.dropna()
    big_gaps = diffs[diffs > 10]  # more than 10 calendar days between rows
    return big_gaps


# ------------------------------------------------------------------
# MAIN
# ------------------------------------------------------------------
for asset_name, path in CSV_PATHS.items():
    print("\n" + "=" * 80)
    print(f"{asset_name}  —  {path}")
    print("=" * 80)

    try:
        df = load_and_clean(path)
    except FileNotFoundError:
        print(f"  FILE NOT FOUND: {path}  (fix CSV_PATHS above)")
        continue
    except Exception as e:
        print(f"  ERROR loading file: {e}")
        continue

    print(f"Total rows: {len(df)}")
    print(f"Date range: {df['Date'].min().date()}  ->  {df['Date'].max().date()}")
    print(f"Close price range (whole file): {df['Close'].min():.2f} - {df['Close'].max():.2f}")

    gaps = check_date_gaps(df)
    if len(gaps) > 0:
        print(f"\n{len(gaps)} gap(s) of >10 calendar days between consecutive rows "
              f"(largest: {gaps.max()} days) — check these aren't hiding missing history.")
    else:
        print("\nNo large gaps (>10 days) between consecutive rows.")

    print("\nPer-regime breakdown (DF1-DF4 as currently defined in the notebook):")
    print(f"{'Regime':<6} {'Years':<12} {'Obs':>6} {'MeanRet%':>10} {'StdRet%':>9} "
          f"{'Skew':>7} {'Kurt':>7} {'AC(1)':>7} {'MaxUp%':>8} {'MaxDown%':>9}")

    for regime_name, (start, end) in REGIMES.items():
        end_label = end if end is not None else df["Date"].dt.year.max()
        regime_df = regime_slice(df, start, end)
        stats = character_stats(regime_df)

        if stats is None:
            print(f"{regime_name:<6} {start}-{end_label:<7} "
                  f"{len(regime_df):>6}   -- insufficient data --")
            continue

        print(f"{regime_name:<6} {start}-{end_label:<7} "
              f"{stats['n_obs']:>6} "
              f"{stats['mean_daily_return_%']:>10.4f} "
              f"{stats['std_daily_return_%']:>9.4f} "
              f"{stats['skewness']:>7.2f} "
              f"{stats['kurtosis']:>7.2f} "
              f"{stats['lag1_autocorr']:>7.3f} "
              f"{stats['max_up_move_%']:>8.2f} "
              f"{stats['max_down_move_%']:>9.2f}")

    print("\nHow to read this once you have numbers:")
    print("  - Higher StdRet% and Kurtosis, more extreme MaxUp/MaxDown  -> more shock-prone.")
    print("  - Higher AC(1) (closer to +1)                              -> smoother / more trend-persistent.")
    print("  - Compare regimes WITHIN an asset to see which one is genuinely the hardest,")
    print("    and compare assets to each other to check the 'Gold is smoothest, Crude Oil")
    print("    is most shock-driven' claim against real numbers instead of assumption.")

print("\n" + "=" * 80)
print("Done. If any asset's actual max date is well before the present day, or price")
print("range looks off vs. what you expect for that asset today, flag it — it means the")
print("'to_present' / 'to_now' filenames are misleading and DF3/DF4 regime windows in")
print("the notebook don't mean what their labels say.")
print("=" * 80)


Gold  —  gold_2007_to_present.csv
Total rows: 1879
Date range: 2007-01-02  ->  2025-12-09
Close price range (whole file): 604.90 - 4053.90

227 gap(s) of >10 calendar days between consecutive rows (largest: 23.0 days) — check these aren't hiding missing history.

Per-regime breakdown (DF1-DF4 as currently defined in the notebook):
Regime Years           Obs   MeanRet%   StdRet%    Skew    Kurt   AC(1)   MaxUp%  MaxDown%
DF1    2007-2011       499     0.6620    9.4137   -1.03    1.03  -0.199    22.32    -26.72
DF2    2012-2019       794     0.1807    6.5677    1.70    7.33  -0.119    36.93    -17.43
DF3    2020-2022       298     0.2243    6.0304   -0.22    1.15  -0.104    21.17    -19.49
DF4    2023-2025       288     0.7450    9.6011   -1.74    3.74  -0.202    18.09    -33.71

How to read this once you have numbers:
  - Higher StdRet% and Kurtosis, more extreme MaxUp/MaxDown  -> more shock-prone.
  - Higher AC(1) (closer to +1)                              -> smoother / more trend-pe

C:\Users\ASUS\AppData\Local\Temp\ipykernel_40020\3983933708.py:54: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
